In [ ]:
DEG_DIR = "../differential_expression/primary-cohort/deseq2_results_batch"
GSEA_DIR = "gsea_results"
FIG_DIR = "figures"
FIG_FORMAT = "png"
FIG_DPI = 600

In [ ]:
import gseapy as gp
import pandas as pd
import numpy as np
import scanpy as sc
import anndata as ad
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D
import colorsys
from adjustText import adjust_text
from itertools import product
import os

os.makedirs(GSEA_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

# Same figure style as differential_expression/primary-cohort/deseq2.ipynb,
# so GSEA and DEG figures read as one consistent set.
plt.rcParams.update({
    "font.family":        "sans-serif",
    "font.sans-serif":    ["Arial", "Helvetica", "DejaVu Sans"],
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.linewidth":     0.8,
    "xtick.major.size":   3,
    "ytick.major.size":   3,
    "xtick.labelsize":    7,
    "ytick.labelsize":    7,
    "axes.titlesize":     8,
    "axes.titleweight":   "bold",
    "axes.labelsize":     7,
    "legend.fontsize":    7,
    "legend.title_fontsize": 8,
    "figure.titlesize":   13,
    "figure.titleweight": "bold",
    "figure.dpi":         300,
})

In [ ]:
def get_sig(results_df, padj=0.05, lfc=1.0):
    """Filter DESeq2 results to significant DEGs."""
    return results_df[
        (results_df["padj"] < padj) &
        (results_df["log2FoldChange"].abs() > lfc)
    ].sort_values("padj")


def load_contrast(name, output_dir=DEG_DIR):
    """
    Load full, shrunk and significant CSVs for one contrast.
    Returns a dict with keys 'full', 'shrunk', 'significant'.
    Falls back to 'full' for the shrunk key if no _shrunk.csv exists
    (e.g. if the run used shrink_coeff=None).
    """
    if name.startswith("condition_condition_in_"):
        name = name.replace("condition_condition_in_", "condition_in_class_")
    name_clean = name.replace(" ", "_").replace(":", "_")
    full_path = os.path.join(output_dir, f"{name_clean}_full.csv")
    sig_path = os.path.join(output_dir, f"{name_clean}_significant.csv")
    shrunk_path = os.path.join(output_dir, f"{name_clean}_shrunk.csv")
                
    if not os.path.exists(full_path):
        print(f"  WARNING: {full_path} not found — skipping")
        return None

    result = {
        "full":        pd.read_csv(full_path, index_col=0),
        "significant": pd.read_csv(sig_path,  index_col=0),
    }
    if os.path.exists(shrunk_path):
        result["shrunk"] = pd.read_csv(shrunk_path, index_col=0)
    else:
        # No separate shrunk file — use full (unshrunk) LFCs for plotting
        result["shrunk"] = result["full"]

    return result


summary_path = os.path.join(DEG_DIR, "summary.csv")
summary_df   = pd.read_csv(summary_path)

print(f"Found {len(summary_df)} contrasts:\n")
print(summary_df.to_string(index=False))

In [ ]:
# ── Load all contrasts ────────────────────────────────────────────────────────
results = {}
for name in summary_df["comparison"]:
    loaded = load_contrast(name)
    if loaded is not None:
        results[name] = loaded

print(f"\nLoaded {len(results)} contrasts successfully.")

In [ ]:
# Condition overall
res_condition = results["condition_short_vs_long"]["full"]
res_condition_shrunk = results["condition_short_vs_long"]["shrunk"]

# Histology pairwise contrasts
res_histology = {
    name.replace("histology_", ""): data["full"]
    for name, data in results.items()
    if name.startswith("histology_")
}
res_histology_shrunk = {
    name.replace("histology_", ""): data["shrunk"]
    for name, data in results.items()
    if name.startswith("histology_")
}

# Class pairwise contrasts
res_class = {
    name.replace("class_", ""): data["full"]
    for name, data in results.items()
    if name.startswith("class_")
}
res_class_shrunk = {
    name.replace("class_", ""): data["shrunk"]
    for name, data in results.items()
    if name.startswith("class_")
}


def _match_contrast(name, prefixes, allowed_suffixes=None):
    """
    Strip whichever of `prefixes` `name` starts with and return the
    remainder, optionally restricted to an exact set of allowed values —
    used below so a longer, more specific contrast name sharing a prefix
    with a shorter one (e.g. "..._Ovary_tumor_epithelium" vs "..._Ovary")
    doesn't get swept into the wrong bucket.
    """
    for prefix in prefixes:
        if name.startswith(prefix):
            suffix = name[len(prefix):]
            if allowed_suffixes is None or suffix in allowed_suffixes:
                return suffix
    return None


SITE_LEVELS = ["Ovary", "Omentum"]

# Condition within each site/class. Older runs of run_deseq2.py saved this
# as "condition_condition_in_{cls}"; the current script saves it as
# "condition_in_class_{cls}" — match both.
res_condition_in_class = {}
res_condition_in_class_shrunk = {}
for name, data in results.items():
    key = _match_contrast(
        name, ["condition_condition_in_", "condition_in_class_"], SITE_LEVELS
    )
    if key is not None:
        res_condition_in_class[key] = data["full"]
        res_condition_in_class_shrunk[key] = data["shrunk"]

# Condition within each site, restricted to Tumor Epithelium only
res_condition_in_class_tumor_epi = {}
res_condition_in_class_tumor_epi_shrunk = {}
for name, data in results.items():
    key = _match_contrast(
        name, ["condition_in_class_"],
        [f"{site}_tumor_epithelium" for site in SITE_LEVELS],
    )
    if key is not None:
        site = key.replace("_tumor_epithelium", "")
        res_condition_in_class_tumor_epi[site] = data["full"]
        res_condition_in_class_tumor_epi_shrunk[site] = data["shrunk"]

# Condition within each histology (saved as condition_in_histology_{hist})
res_condition_in_histology = {
    name.replace("condition_in_histology_", ""): data["full"]
    for name, data in results.items()
    if name.startswith("condition_in_histology_")
}
res_condition_in_histology_shrunk = {
    name.replace("condition_in_histology_", ""): data["shrunk"]
    for name, data in results.items()
    if name.startswith("condition_in_histology_")
}

print("res_condition                       :", "loaded" if res_condition is not None else "NOT FOUND")
print("res_histology                       :", list(res_histology.keys()))
print("res_class                           :", list(res_class.keys()))
print("res_condition_in_class              :", list(res_condition_in_class.keys()))
print("res_condition_in_class_tumor_epi    :", list(res_condition_in_class_tumor_epi.keys()))
print("res_condition_in_histology          :", list(res_condition_in_histology.keys()))

In [ ]:
names = gp.get_library_name()
names

In [ ]:
import requests
import os

def download_gmt_direct(gene_sets, outdir):
    """
    Download GMT files directly from the Enrichr API, bypassing
    gseapy's get_library() which fails to parse certain libraries.
    """
    os.makedirs(outdir, exist_ok=True)
    downloaded = {}

    base_url = "https://maayanlab.cloud/Enrichr/geneSetLibrary?mode=text&libraryName={}"

    for gs in gene_sets:
        gmt_path = os.path.join(outdir, f"{gs}.gmt")

        if os.path.exists(gmt_path):
            print(f"  Already exists: {gmt_path}")
            downloaded[gs] = gmt_path
            continue

        print(f"  Downloading: {gs}")
        try:
            r = requests.get(base_url.format(gs), timeout=60)
            r.raise_for_status()

            with open(gmt_path, "w") as f:
                f.write(r.text)

            # Count lines as a sanity check
            n_sets = len([l for l in r.text.strip().split("\n") if l.strip()])
            print(f"    Saved {n_sets} gene sets → {gmt_path}")
            downloaded[gs] = gmt_path

        except Exception as e:
            print(f"    Failed: {gs}: {e}")

    return downloaded

In [ ]:
# ── Gene set databases to test ────────────────────────────────────────────────
GENE_SETS = [
    "MSigDB_Hallmark_2020",
    # "KEGG_2021_Human",
    # "Reactome_Pathways_2024",
    # "GO_Biological_Process_2026",
]
downloaded_gmts = download_gmt_direct(GENE_SETS, GSEA_DIR)

GSEA_THREADS  = 4
GSEA_PERMUTATIONS = 1000
MIN_SIZE      = 15    # minimum gene set size
MAX_SIZE      = 500   # maximum gene set size


def build_ranking(results_df):
    """
    Build a ranked gene list from DESeq2 results for pre-ranked GSEA.

    Ranking metric: sign(log2FC) * -log10(pvalue)
    - Preserves direction of effect
    - Weights by significance rather than effect size alone
    - Genes with NA pvalue (low-count filtering by DESeq2) are dropped
    """
    df = results_df.dropna(subset=["pvalue", "log2FoldChange"]).copy()

    # Clip p-values to avoid -log10(0) = inf
    df["pvalue_clipped"] = df["pvalue"].clip(lower=1e-300)

    df["rank_metric"] = (
        np.sign(df["log2FoldChange"]) *
        -np.log10(df["pvalue_clipped"])
    )

    # Sort descending — strongly upregulated genes at top,
    # strongly downregulated at bottom
    return df["rank_metric"].sort_values(ascending=False)


def run_preranked_gsea(ranking, label, gene_sets=GENE_SETS, outdir=GSEA_DIR):
    """
    Run pre-ranked GSEA for a given ranked gene list against multiple
    gene set databases and save results to CSV.
    """
    label_clean = label.replace(" ", "_").replace("/", "_")
    results     = {}

    for gs in gene_sets:
        print(f"  Running GSEA: {label} — {gs}")

        gmt_path = os.path.join(outdir, f"{gs}.gmt")
        gene_set_input = gmt_path if os.path.exists(gmt_path) else gs
        max_size = len(ranking)

        if os.path.exists(gmt_path):
            gmt_genes = set()
            with open(gmt_path) as f:
                for line in f:
                    parts = line.strip().split("\t")
                    gmt_genes.update(parts[2:])   # skip name and description
            overlap = len(set(ranking.index) & gmt_genes)
            print(f"    Gene overlap with GMT: {overlap} / {len(ranking.index)} "
                  f"ranking genes, {len(gmt_genes)} GMT genes")
            if overlap < MIN_SIZE:
                print(f"    WARNING: overlap ({overlap}) < min_size ({MIN_SIZE}) "
                      f"— GSEA will fail. Check gene name format in GMT vs ranking.")
                continue
        
        try:
            pre_res = gp.prerank(
                rnk=ranking,
                gene_sets=gene_set_input,
                threads=GSEA_THREADS,
                min_size=MIN_SIZE,
                max_size=max_size,
                permutation_num=GSEA_PERMUTATIONS,
                outdir=None,
                seed=42,
                verbose=False,
            )

            res_df = pre_res.res2d.copy()

            # Save full results
            out_path = os.path.join(
                outdir, f"gsea_{label_clean}_{gs}.csv"
            )
            res_df.to_csv(out_path, index=False)

            # Print significant pathways
            sig = res_df[res_df["FDR q-val"] < 0.25]  # standard GSEA FDR threshold
            print(f"    Significant pathways (FDR<0.25): {len(sig)}")
            if len(sig) > 0:
                print(sig[["Term", "NES", "NOM p-val", "FDR q-val"]]
                      .head(10).to_string(index=False))

            results[gs] = res_df

        except Exception as e:
            print(f"    GSEA failed for {gs}: {e}")

    return results

## Run GSEA — three separate analysis families

Each analysis family answers a different question and is run and plotted independently.

1. **Overall** — does the PFI effect (short vs medium & long) show up at all,
   pooling every sample?
2. **By site** — is that PFI effect consistent between the adnexal tumor and
   the omental metastasis (`Ovary` → *Adnexa*, `Omentum` → *Omentum*)?
   - **2b. By site, Tumor Epithelium only** — the same site comparison, but
     isolated to a single histological compartment so it isn't confounded by
     adnexa and omentum having different tumor/stroma proportions.
3. **By histology** — is that PFI effect consistent between tumor epithelium
   and stroma (`Tumor Epithelium` → *Tumor*, `Other` → *Stroma*)?

In [ ]:
# ── Family 1: overall PFI effect (short vs medium & long, all samples) ─────
print("═" * 70)
print("PFI effect — overall (short vs medium & long, all samples)")
print("═" * 70)
ranking_overall = build_ranking(res_condition)
print(f"  Genes in ranking: {len(ranking_overall):,}")
print(f"  Top 5 upregulated in short:        {ranking_overall.head().index.tolist()}")
print(f"  Top 5 upregulated in medium/long:  {ranking_overall.tail().index.tolist()}")

gsea_overall = run_preranked_gsea(ranking_overall, label="pfi_overall")
ranking_overall.to_csv(os.path.join(GSEA_DIR, "gene_ranking_overall.csv"), index=True)

# ── Family 2: PFI effect within each site (adnexa vs omentum) ──────────────
print("\n" + "═" * 70)
print("PFI effect — by site (adnexa vs omentum)")
print("═" * 70)

SITE_LABELS = {"Ovary": "Adnexa", "Omentum": "Omentum"}
gsea_by_site = {}
for site_key, site_label in SITE_LABELS.items():
    if site_key not in res_condition_in_class:
        print(f"  '{site_key}' not found in res_condition_in_class — skipping")
        continue
    print(f"\n  ── Site: {site_label} ──")
    ranking = build_ranking(res_condition_in_class[site_key])
    print(f"    Genes in ranking: {len(ranking):,}")
    gsea_by_site[site_label] = run_preranked_gsea(
        ranking, label=f"pfi_by_site_{site_label}"
    )


# ── Family 2b: PFI effect by site, restricted to Tumor Epithelium ──────────
# Same site comparison as Family 2, but isolated to a single histological
# compartment so the site effect isn't confounded by differing tumor/stroma
# proportions between adnexa and omentum.
print("\n" + "═" * 70)
print("PFI effect — by site, Tumor Epithelium only")
print("═" * 70)

gsea_by_site_tumor_epi = {}
for site_key, site_label in SITE_LABELS.items():
    if site_key not in res_condition_in_class_tumor_epi:
        print(f"  '{site_key}' not found in res_condition_in_class_tumor_epi "
              f"— skipping (re-run run_deseq2.py to generate this contrast)")
        continue
    print(f"\n  ── Site: {site_label} (Tumor Epithelium only) ──")
    ranking = build_ranking(res_condition_in_class_tumor_epi[site_key])
    print(f"    Genes in ranking: {len(ranking):,}")
    gsea_by_site_tumor_epi[site_label] = run_preranked_gsea(
        ranking, label=f"pfi_by_site_tumor_epi_{site_label}"
    )


# ── Family 3: PFI effect within each histology (tumor vs stroma) ───────────
print("\n" + "═" * 70)
print("PFI effect — by histology (tumor vs stroma)")
print("═" * 70)

HISTOLOGY_LABELS = {"Tumor Epithelium": "Tumor", "Other": "Stroma"}
gsea_by_histology = {}
for hist_key, hist_label in HISTOLOGY_LABELS.items():
    if hist_key not in res_condition_in_histology:
        print(f"  '{hist_key}' not found in res_condition_in_histology — skipping")
        continue
    print(f"\n  ── Histology: {hist_label} ──")
    ranking = build_ranking(res_condition_in_histology[hist_key])
    print(f"    Genes in ranking: {len(ranking):,}")
    gsea_by_histology[hist_label] = run_preranked_gsea(
        ranking, label=f"pfi_by_histology_{hist_label}"
    )


# ── Summary: significant pathways across all families ──────────────────────
print("\n── GSEA summary (FDR < 0.25) ──")

def _collect_gsea(gsea_results, contrast_label, family, rows):
    for gs, res_df in gsea_results.items():
        sig = res_df[res_df["FDR q-val"] < 0.25]
        for _, row in sig.iterrows():
            rows.append({
                "family":      family,
                "contrast":    contrast_label,
                "gene_set_db": gs,
                "term":        row["Term"],
                "NES":         row["NES"],
                "nom_pval":    row["NOM p-val"],
                "fdr_qval":    row["FDR q-val"],
            })

summary_rows = []
_collect_gsea(gsea_overall, "All samples", "overall", summary_rows)
for label, res in gsea_by_site.items():
    _collect_gsea(res, label, "by_site", summary_rows)
for label, res in gsea_by_site_tumor_epi.items():
    _collect_gsea(res, label, "by_site_tumor_epithelium", summary_rows)
for label, res in gsea_by_histology.items():
    _collect_gsea(res, label, "by_histology", summary_rows)

summary_gsea = pd.DataFrame(summary_rows).sort_values(["family", "fdr_qval"])
summary_gsea.to_csv(os.path.join(GSEA_DIR, "gsea_summary.csv"), index=False)
print(f"Total significant pathways: {len(summary_gsea)}")
print(summary_gsea.head(20).to_string(index=False))

In [ ]:
# ── CONFIG for visualization ────────────────────────────────────────────────
FDR_THRESH   = 0.25   # standard GSEA FDR cutoff for "significant"
NOM_P_THRESH = 0.05   # nominal p-value cutoff for significance stars

CONTRASTS_OVERALL          = {"All samples": gsea_overall}
CONTRASTS_BY_SITE          = gsea_by_site
CONTRASTS_BY_SITE_TUM_EPI  = gsea_by_site_tumor_epi
CONTRASTS_BY_HISTOLOGY     = gsea_by_histology

# Distinct palettes per analysis type, so a "by site" figure and a "by
# histology" figure can never be mistaken for one another at a glance —
# blue/orange/red for site comparisons, green/purple/teal for histology.
SITE_COLORS = {
    "not_significant": "#D9D9D9",
    "x_only":          "#4C78A8",   # steel blue
    "y_only":          "#F58518",   # orange
    "both":            "#B22222",   # firebrick red
}
HISTOLOGY_COLORS = {
    "not_significant": "#D9D9D9",
    "x_only":          "#59A14F",   # green
    "y_only":          "#B07AA1",   # mauve / purple
    "both":            "#117864",   # dark teal
}

In [ ]:
def annotate_heatmap(data, pvals, ax, **kwargs):
    """
    Annotate heatmap cells with significance stars based on p-values.
    *** p < 0.001 / ** p < 0.01 / * p < 0.05
    """
    for i, j in product(range(data.shape[0]), range(data.shape[1])):
        pval = pvals.iloc[i, j]
        if pval < 0.001:
            stars = "***"
        elif pval < 0.01:
            stars = "**"
        elif pval < 0.05:
            stars = "*"
        else:
            stars = ""
        if stars:
            ax.text(j + 0.5, i + 0.5, stars,
                    ha="center", va="center", **kwargs)


def plot_gsea_heatmap(
    df_nes, df_pval,
    title=None,
    figsize=None,
    save_path=None,
):
    """
    Heatmap of GSEA NES scores with significance star annotations.
    """
    if figsize is None:
        # Auto-size based on number of pathways and contrasts
        figsize = (
            max(4, df_nes.shape[1] * 1.5),
            max(4, df_nes.shape[0] * 0.5),
        )

    palette = sns.diverging_palette(250, 30, l=65, s=80,
                                    center="light", as_cmap=True)

    # Symmetric color scale centered on 0
    abs_max = df_nes.abs().max().max()

    sns.set_style("whitegrid", {"axes.grid": False})
    fig, ax = plt.subplots(figsize=figsize)

    sns.heatmap(
        df_nes,
        cmap=palette,
        center=0,
        vmin=-abs_max,
        vmax=abs_max,
        linewidths=0.4,
        linecolor="#EEEEEE",
        cbar_kws={"shrink": 0.5, "aspect": 15},
        ax=ax,
    )

    # Style colorbar
    cbar = ax.collections[0].colorbar
    cbar.ax.set_ylabel("NES", fontsize=12, labelpad=15, rotation=0)
    cbar.ax.yaxis.set_label_position("right")
    cbar.ax.tick_params(labelsize=10)

    # Significance annotations
    annotate_heatmap(df_nes, df_pval, ax=ax, fontsize=12, color="black")

    if title:
        ax.set_title(title, fontsize=13, pad=12, x=0.2, fontweight="bold")

    ax.set_xlabel(None)
    ax.set_ylabel(None)
    ax.tick_params(axis="x", labelsize=11, rotation=45)
    plt.gcf().canvas.draw()
    for lbl in ax.get_xticklabels():
        lbl.set_ha("right")
        lbl.set_rotation_mode("anchor")
    ax.tick_params(axis="y", labelsize=10, rotation=0)

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, bbox_inches="tight")
        print(f"Saved: {save_path}")

    plt.show()
    return fig, ax

In [ ]:
def build_gsea_heatmap_data_combined(
    contrasts_to_plot,
    gene_set_dbs,
    fdr_thresh=FDR_THRESH,
    n_top=15,
):
    """
    Build NES and p-value DataFrames combining results across one or more
    gene set databases, keeping only the top n_top pathways by mean
    absolute NES across contrasts.

    `contrasts_to_plot` keys are used directly as the output column labels,
    so pass a dict that's already keyed by the display name you want (e.g.
    {"Adnexa": ..., "Omentum": ...}) — no renaming happens here.
    """
    def clean_pathway(name):
        for prefix in ["HALLMARK_", "KEGG_", "REACTOME_", "GOBP_"]:
            name = name.replace(prefix, "")
        return name.replace("_", " ").title()

    # Short display names for the database label in parentheses
    db_short_names = {
        "MSigDB_Hallmark_2020":       "Hallmark",
        "KEGG_2021_Human":            "KEGG",
        "Reactome_2022":              "Reactome",
        "GO_Biological_Process_2023": "GO:BP",
    }
    # Only qualify pathway names with the db label when combining more than
    # one database — keeps the common single-db case clean.
    tag_db = len(gene_set_dbs) > 1

    # ── Collect all significant pathways across databases and contrasts ────────
    all_nes  = {}
    all_pval = {}
    all_fdr  = {}

    for db in gene_set_dbs:
        db_label = db_short_names.get(db, db)

        for contrast_label, gsea_results in contrasts_to_plot.items():
            if db not in gsea_results:
                continue

            df = gsea_results[db].set_index("Term")

            # Only consider pathways significant in this contrast
            sig = df[df["FDR q-val"] < fdr_thresh]

            for pathway, row in sig.iterrows():
                clean_name = clean_pathway(pathway)
                if tag_db:
                    clean_name = f"{clean_name} ({db_label})"

                if clean_name not in all_nes:
                    all_nes[clean_name]  = {}
                    all_pval[clean_name] = {}
                    all_fdr[clean_name]  = {}

                all_nes[clean_name][contrast_label]  = row["NES"]
                all_pval[clean_name][contrast_label] = row["NOM p-val"]
                all_fdr[clean_name][contrast_label]  = row["FDR q-val"]

    if not all_nes:
        print(f"No pathways significant at FDR < {fdr_thresh} in any contrast")
        return None, None, None

    # ── Build full matrices, filling missing with neutral values ──────────────
    contrast_cols = list(contrasts_to_plot.keys())

    df_nes  = pd.DataFrame(all_nes,  index=contrast_cols).T.fillna(0)
    df_pval = pd.DataFrame(all_pval, index=contrast_cols).T.fillna(1)
    df_fdr  = pd.DataFrame(all_fdr,  index=contrast_cols).T.fillna(1)

    print(f"Total significant pathways across all databases: {len(df_nes)}")

    # ── Select top n_top by mean absolute NES ────────────────────────────────
    mean_abs_nes = df_nes.abs().mean(axis=1).sort_values(ascending=False)
    top_pathways = mean_abs_nes.head(n_top).index

    df_nes  = df_nes.loc[top_pathways]
    df_pval = df_pval.loc[top_pathways]
    df_fdr  = df_fdr.loc[top_pathways]

    # Sort by mean NES (signed) so up/down pathways cluster together
    df_nes  = df_nes.loc[df_nes.mean(axis=1).sort_values(ascending=False).index]
    df_pval = df_pval.loc[df_nes.index]
    df_fdr  = df_fdr.loc[df_nes.index]

    print(f"Showing top {len(df_nes)} pathways by mean absolute NES")

    return df_nes, df_pval, df_fdr

In [ ]:
def plot_gsea_bar(df_nes, df_pval, column, title=None, figsize=None, save_path=None):
    """
    Horizontal bar chart of NES for a single contrast column (as produced
    by build_gsea_heatmap_data_combined). More readable than a 1-column
    heatmap when there's only one comparison to show.
    """
    nes  = df_nes[column].sort_values()
    pval = df_pval[column].reindex(nes.index)

    if figsize is None:
        figsize = (5, max(2, 0.35 * len(nes)))

    colors = ["#C0392B" if v > 0 else "#2E5A87" for v in nes]

    fig, ax = plt.subplots(figsize=figsize)
    ax.barh(nes.index, nes.values, color=colors)
    ax.axvline(0, color="black", linewidth=0.8)

    pad = max(nes.abs().max() * 0.04, 0.03)
    for y, (v, p) in enumerate(zip(nes.values, pval.values)):
        stars = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
        if stars:
            ax.text(v + (pad if v > 0 else -pad), y, stars,
                    ha="left" if v > 0 else "right", va="center", fontsize=10)

    ax.set_xlim(-3,3)
    ax.set_xlabel("NES", fontsize=11)
    if title:
        ax.set_title(title, fontsize=11, fontweight="bold", pad=10, loc='left')
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, bbox_inches="tight")
        print(f"Saved: {save_path}")
    plt.show()
    return fig, ax


def darken_color(color, factor=0.7):
    """Darken a matplotlib color by reducing its lightness."""
    rgb = mcolors.to_rgb(color)
    h, l, s = colorsys.rgb_to_hls(*rgb)
    l = max(0, l * factor)
    return colorsys.hls_to_rgb(h, l, s)


def clean_pathway_name(name):
    for prefix in ["HALLMARK_", "KEGG_", "REACTOME_", "GOBP_"]:
        name = name.replace(prefix, "")
    return name.replace("_", " ").title()


def plot_gsea_effect_comparison(
    gsea_x, gsea_y, gene_set_db, x_label, y_label,
    pathways_of_interest=None,
    fdr_thresh=FDR_THRESH,
    colors=SITE_COLORS,
    title=None,
    figsize=(6.5, 6.5),
    label_fontsize=9,
    point_size=35,
    poi_point_size=120,
    n_top_specific=5,
    n_top_both=3,
    save_path=None,
):
    """
    Scatter plot comparing GSEA NES between two contrasts for one gene set
    database — the GSEA analog of plot_effect_comparison() in
    differential_expression/primary-cohort/deseq2.ipynb, same "X only / Y
    only / Both / Not significant" quadrant coloring and top-divergent
    callouts, applied to NES instead of log2FC.
    """
    if gene_set_db not in gsea_x or gene_set_db not in gsea_y:
        print(f"  WARNING: {gene_set_db} missing for {x_label} or {y_label}")
        return None, None, None

    pathways_of_interest = pathways_of_interest or []

    # ── Merge ────────────────────────────────────────────────────────────────
    # Outer join, not inner: GSEA is run independently per contrast, so a
    # pathway that fell below MIN_SIZE gene overlap in one group but not the
    # other is treated as NES=0 / not significant there, rather than dropped.
    df_x = gsea_x[gene_set_db].set_index("Term")[["NES", "FDR q-val"]] \
        .rename(columns={"NES": "nes_x", "FDR q-val": "fdr_x"})
    df_y = gsea_y[gene_set_db].set_index("Term")[["NES", "FDR q-val"]] \
        .rename(columns={"NES": "nes_y", "FDR q-val": "fdr_y"})
    merged = df_x.join(df_y, how="outer")
    merged[["nes_x", "nes_y"]] = merged[["nes_x", "nes_y"]].fillna(0)
    merged[["fdr_x", "fdr_y"]] = merged[["fdr_x", "fdr_y"]].fillna(1)

    # ── Significance classification ─────────────────────────────────────────
    merged["sig_x"] = merged["fdr_x"] < fdr_thresh
    merged["sig_y"] = merged["fdr_y"] < fdr_thresh

    def classify(row):
        if row["sig_x"] and row["sig_y"]:   return "Both"
        elif row["sig_x"]:                  return f"{x_label} only"
        elif row["sig_y"]:                  return f"{y_label} only"
        else:                               return "Not significant"

    merged["group"] = merged.apply(classify, axis=1)

    color_map = {
        "Not significant":  colors["not_significant"],
        f"{x_label} only":  colors["x_only"],
        f"{y_label} only":  colors["y_only"],
        "Both":             colors["both"],
    }

    # ── Most group-specific pathways ────────────────────────────────────────
    top_x_only = (
        merged[merged["group"] == f"{x_label} only"]
        .assign(rank_metric=lambda d: d["nes_x"].abs())
        .nlargest(n_top_specific, "rank_metric")
        .index.tolist()
    )
    top_y_only = (
        merged[merged["group"] == f"{y_label} only"]
        .assign(rank_metric=lambda d: d["nes_y"].abs())
        .nlargest(n_top_specific, "rank_metric")
        .index.tolist()
    )
    # Top enriched pathways significant in both — labeled on the plot only,
    # no separate legend entry (they're still plotted in the "Both" color).
    top_both = (
        merged[merged["group"] == "Both"]
        .assign(rank_metric=lambda d: (d["nes_x"].abs() + d["nes_y"].abs()) / 2)
        .nlargest(n_top_both, "rank_metric")
        .index.tolist()
    )

    poi_set = set(pathways_of_interest)
    specific_pathways = {
        pw: group
        for group, pws in [
            (f"{x_label} only", top_x_only),
            (f"{y_label} only", top_y_only),
            ("Both", top_both),
        ]
        for pw in pws
        if pw not in poi_set
    }

    # ── Figure ───────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=figsize)

    for group in ["Not significant", f"{x_label} only", f"{y_label} only", "Both"]:
        sub = merged[merged["group"] == group]
        ax.scatter(
            sub["nes_x"], sub["nes_y"],
            s=point_size,
            c=color_map[group],
            alpha=0.75 if group != "Not significant" else 0.45,
            edgecolors="none",
            zorder=2 if group != "Not significant" else 1,
        )

    ax.axhline(0, color="black", linestyle="--", linewidth=0.9, zorder=0)
    ax.axvline(0, color="black", linestyle="--", linewidth=0.9, zorder=0)

    finite_vals = np.r_[merged["nes_x"].values, merged["nes_y"].values]
    finite_vals = finite_vals[np.isfinite(finite_vals)]
    lim = np.nanmax(np.abs(finite_vals))
    lim = max(lim, 1.0)
    lim = np.ceil(lim * 1.05 * 2) / 2
    ax.plot([-lim, lim], [-lim, lim],
            linestyle=":", linewidth=1.2, color="#666666", zorder=0)

    # ── Annotate specific pathways ──────────────────────────────────────────
    specific_color_map = {
        f"{x_label} only": color_map[f"{x_label} only"],
        f"{y_label} only": color_map[f"{y_label} only"],
        "Both":            color_map["Both"],
    }

    texts = []
    for pw, group in specific_pathways.items():
        if pw not in merged.index:
            continue
        row   = merged.loc[pw]
        color = specific_color_map[group]
        # Only circle groups that have a matching legend entry
        if group != "Both":
            ax.scatter(
                row["nes_x"], row["nes_y"],
                s=poi_point_size * 0.8,
                facecolors="none",
                edgecolors=color,
                linewidths=1.5,
                zorder=4,
            )
        ax.scatter(
            row["nes_x"], row["nes_y"],
            s=poi_point_size * 0.8,
            facecolors="none",
            edgecolors=color,
            linewidths=1.5,
            zorder=4,
        )
        txt = ax.text(
            row["nes_x"], row["nes_y"], clean_pathway_name(pw),
            fontsize=label_fontsize - 1,
            color=color,
            fontstyle="italic",
            zorder=5,
        )
        txt.set_path_effects([pe.withStroke(linewidth=2, foreground="white")])
        texts.append(txt)

    # ── Annotate pathways of interest ───────────────────────────────────────
    poi_present = [p for p in pathways_of_interest if p in merged.index]
    poi_missing = [p for p in pathways_of_interest if p not in merged.index]
    if poi_missing:
        print(f"  Pathways of interest not found: {poi_missing}")

    if poi_present:
        poi_df     = merged.loc[poi_present].copy()
        poi_colors = poi_df["group"].map(
            lambda g: darken_color(color_map.get(g, "#888888"), 0.8)
        )
        ax.scatter(
            poi_df["nes_x"], poi_df["nes_y"],
            s=poi_point_size,
            c=poi_colors,
            edgecolors="black",
            linewidths=0.9,
            zorder=5,
        )
        for pw, row in poi_df.iterrows():
            txt = ax.text(
                row["nes_x"], row["nes_y"], clean_pathway_name(pw),
                fontsize=label_fontsize,
                color="black",
                fontweight="bold",
                zorder=6,
            )
            txt.set_path_effects([pe.withStroke(linewidth=2.5, foreground="white")])
            texts.append(txt)

    adjust_text(
        texts, ax=ax,
        expand=(1.15, 1.25),
        arrowprops=dict(arrowstyle="-", lw=0.6, color="#888888",
                       shrinkA=3, shrinkB=3),
    )

    # ── Styling ──────────────────────────────────────────────────────────────
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel(f"NES — {x_label}", fontsize=11)
    ax.set_ylabel(f"NES — {y_label}", fontsize=11)
    ax.set_title(title or f"GSEA effect comparison — {x_label} vs {y_label}",
                 fontsize=11, pad=15, loc="left")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", width=0.9)

    # ── Legend ───────────────────────────────────────────────────────────────
    legend_elements = [
        Line2D([0],[0], marker="o", color="w", label="Not significant",
               markerfacecolor=color_map["Not significant"], markersize=6),
        Line2D([0],[0], marker="o", color="w", label=f"{x_label} only",
               markerfacecolor=color_map[f"{x_label} only"], markersize=6),
        Line2D([0],[0], marker="o", color="w", label=f"{y_label} only",
               markerfacecolor=color_map[f"{y_label} only"], markersize=6),
        Line2D([0],[0], marker="o", color="w", label="Both",
               markerfacecolor=color_map["Both"], markersize=6),
        Line2D([0],[0], marker="o", color="w",
               label=f"Top specific to {x_label}",
               markerfacecolor="none",
               markeredgecolor=color_map[f"{x_label} only"],
               markeredgewidth=1.5, markersize=7),
        Line2D([0],[0], marker="o", color="w",
               label=f"Top specific to {y_label}",
               markerfacecolor="none",
               markeredgecolor=color_map[f"{y_label} only"],
               markeredgewidth=1.5, markersize=7),
        Line2D([0],[0], marker="o", color="w",
               label="Top specific in Both",
               markerfacecolor="none",
               markeredgecolor=color_map["Both"],
               markeredgewidth=1.5, markersize=7),
    ]
    if poi_present:
        legend_elements.append(
            Line2D([0],[0], marker="o", color="w", label="Pathway of interest",
                   markerfacecolor="#C00000", markeredgecolor="black", markersize=7)
        )
    ax.legend(
        handles=legend_elements,
        loc="best",
        frameon=True,
        facecolor="white",
        edgecolor="#CCCCCC",
        framealpha=0.92,
        borderpad=0.8,
        fontsize=9,
    )

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=FIG_DPI, bbox_inches="tight")
        print(f"Saved: {save_path}")
    plt.show()

    return fig, ax, merged

## Visualize results

One figure per family below, plus a divergence scatter for each stratified
family that shows exactly which pathways behave differently between the two
groups — rather than placing site and histology columns in the same table
and leaving the reader to guess whether they're comparable.

### 1. Overall PFI effect (short vs medium & long, all samples)

In [ ]:
df_nes_overall, df_pval_overall, _ = build_gsea_heatmap_data_combined(
    contrasts_to_plot=CONTRASTS_OVERALL,
    gene_set_dbs=GENE_SETS,
    fdr_thresh=FDR_THRESH,
    n_top=15,
)

if df_nes_overall is not None:
    plot_gsea_bar(
        df_nes_overall, df_pval_overall, column="All samples",
        title="All TMA cores",
        figsize=(5,5),
        save_path=os.path.join(FIG_DIR, f"gsea_bar_overall.{FIG_FORMAT}"),
    )

### 2. PFI effect by site (adnexa vs omentum)

In [ ]:
df_nes_site, df_pval_site, _ = build_gsea_heatmap_data_combined(
    contrasts_to_plot=CONTRASTS_BY_SITE,
    gene_set_dbs=GENE_SETS,
    fdr_thresh=FDR_THRESH,
    n_top=15,
)

if df_nes_site is not None:
    plot_gsea_heatmap(
        df_nes_site, df_pval_site,
        title="By site\n(adnexa vs omentum)",
        save_path=os.path.join(FIG_DIR, f"gsea_heatmap_by_site.{FIG_FORMAT}"),
    )

for gs in GENE_SETS:
    plot_gsea_effect_comparison(
        gsea_by_site.get("Adnexa", {}), gsea_by_site.get("Omentum", {}), gs,
        n_top_both=5,
        x_label="Adnexa", y_label="Omentum",
        figsize=(10,5),
        title="By site",
        save_path=os.path.join(FIG_DIR, f"gsea_effect_comparison_site_{gs}.{FIG_FORMAT}"),
    )

### 2b. PFI effect by site, Tumor Epithelium only

Same site comparison as above, restricted to tumor epithelium spots so
differing tumor/stroma proportions between adnexa and omentum can't drive
the site difference. Requires re-running `run_deseq2.py` to generate the
`condition_in_class_{site}_tumor_epithelium` contrasts this depends on.

In [ ]:
df_nes_site_te, df_pval_site_te, _ = build_gsea_heatmap_data_combined(
    contrasts_to_plot=CONTRASTS_BY_SITE_TUM_EPI,
    gene_set_dbs=GENE_SETS,
    fdr_thresh=FDR_THRESH,
    n_top=15,
)

if df_nes_site_te is not None:
    plot_gsea_heatmap(
        df_nes_site_te, df_pval_site_te,
        title="PFI effect by site\n(Tumor Epithelium only)",
        save_path=os.path.join(FIG_DIR, f"gsea_heatmap_by_site_tumor_epi.{FIG_FORMAT}"),
    )

for gs in GENE_SETS:
    plot_gsea_effect_comparison(
        gsea_by_site_tumor_epi.get("Adnexa", {}), gsea_by_site_tumor_epi.get("Omentum", {}), gs,
        x_label="Adnexa", y_label="Omentum",
        colors=SITE_COLORS,
        n_top_both=5,
        title="By site",
        figsize=(10,5),
        save_path=os.path.join(FIG_DIR, f"gsea_effect_comparison_site_tumor_epi_{gs}.{FIG_FORMAT}"),
    )

### 3. PFI effect by histology (tumor vs stroma)

In [ ]:
df_nes_hist, df_pval_hist, _ = build_gsea_heatmap_data_combined(
    contrasts_to_plot=CONTRASTS_BY_HISTOLOGY,
    gene_set_dbs=GENE_SETS,
    fdr_thresh=FDR_THRESH,
    n_top=15,
)

if df_nes_hist is not None:
    plot_gsea_heatmap(
        df_nes_hist, df_pval_hist,
        title="PFI effects by histology\n(tumor vs stroma)",
        save_path=os.path.join(FIG_DIR, f"gsea_heatmap_by_histology.{FIG_FORMAT}"),
    )

for gs in GENE_SETS:
    plot_gsea_effect_comparison(
        gsea_by_histology.get("Tumor", {}), gsea_by_histology.get("Stroma", {}), gs,
        x_label="Tumor", y_label="Stroma",
        colors=HISTOLOGY_COLORS, n_top_both=5,
        title="By tissue compartment",
        figsize=(10,5),
        save_path=os.path.join(FIG_DIR, f"gsea_effect_comparison_histology_{gs}.{FIG_FORMAT}"),
    )